# Assignment 5 — Gradient Descent, Polynomial Regression, Regularization

**SVNIT — Department of Artificial Intelligence**

**Question sheet:** `Assignment-5.pdf`

**Data:** synthetic NumPy (Q1–Q4), UCI Wine Quality red (`../data/winequality-red.csv`, Q5–Q6), Iris (Q7)

`np.random.seed` / `RandomState(42)` is used wherever the question says "random", so the numbers stay the same on a re-run. Plots are saved to `../plots/assignment_5/` and shown under each question.


In [1]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    root_mean_squared_error,
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

WINE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
)


def find_project_root():
    starts = [Path.cwd().resolve()]
    for key in ("__vsc_ipynb_file__", "__session__"):
        val = globals().get(key)
        if val:
            starts.append(Path(val).resolve().parent)
    seen = set()
    for start in starts:
        for p in [start, *start.parents]:
            if p in seen:
                continue
            seen.add(p)
            if (p / "data").is_dir():
                return p
    raise FileNotFoundError("Could not find the ML Lab data/ folder.")


ROOT = find_project_root()
DATA = ROOT / "data"
PLOTS = ROOT / "plots" / "assignment_5"
PLOTS.mkdir(parents=True, exist_ok=True)

wine_path = DATA / "winequality-red.csv"
if not wine_path.exists():
    print("Downloading UCI red wine quality...")
    urlretrieve(WINE_URL, wine_path)


def section(title):
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def save_plot(name):
    path = PLOTS / name
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Saved {path.relative_to(ROOT)}")


def add_bias(X):
    return np.c_[np.ones((len(X), 1)), X]


def mse(X_b, y, theta):
    return float(np.mean((X_b @ theta - y) ** 2))


def first_epoch_below(history, threshold=1.0):
    for epoch, value in enumerate(history, start=1):
        if value < threshold:
            return epoch
    return None


print("Project root:", ROOT)
print("Wine data:", wine_path)


Project root: /Users/ojas/Desktop/ML Lab
Wine data: /Users/ojas/Desktop/ML Lab/data/winequality-red.csv


## Q1: Batch Gradient Descent


In [2]:
section("Q1: Batch Gradient Descent")

# y = 4 + 3x + noise. Closed-form answer is near θ = [4, 3].
rng = np.random.RandomState(42)
X = 2 * rng.rand(100, 1)
y = 4 + 3 * X + rng.randn(100, 1)
X_b = add_bias(X)
theta0 = rng.randn(2, 1)
m = len(X_b)

eta = 0.1
n_iterations = 1000
theta = theta0.copy()
batch_mse = []
for _ in range(n_iterations):
    gradients = (2 / m) * X_b.T @ (X_b @ theta - y)
    theta = theta - eta * gradients
    batch_mse.append(mse(X_b, y, theta))

theta_batch = theta
theta_ols = np.linalg.lstsq(X_b, y, rcond=None)[0]

print(f"Samples: {m}")
print(f"Initial θ (random):  θ0 = {theta0[0, 0]:+.4f}   θ1 = {theta0[1, 0]:+.4f}")
print(f"Batch GD final θ:    θ0 = {theta_batch[0, 0]:+.4f}   θ1 = {theta_batch[1, 0]:+.4f}")
print(f"Normal equation θ:   θ0 = {theta_ols[0, 0]:+.4f}   θ1 = {theta_ols[1, 0]:+.4f}")
print(f"Final training MSE:  {batch_mse[-1]:.4f}")
print(f"Updates: {n_iterations}  (one full-batch step per iteration)")
print("True generating line is θ = [4, 3]. GD matches the normal equation,")
print("so it found the least-squares fit. Noise keeps that fit a bit off [4, 3].")

plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, n_iterations + 1), batch_mse, color="steelblue")
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Batch GD: MSE vs iteration  (η = 0.1)")
save_plot("q1_batch_gd_mse.png")



  Q1: Batch Gradient Descent
Samples: 100
Initial θ (random):  θ0 = +0.0130   θ1 = +1.4535
Batch GD final θ:    θ0 = +4.2151   θ1 = +2.7701
Normal equation θ:   θ0 = +4.2151   θ1 = +2.7701
Final training MSE:  0.8066
Updates: 1000  (one full-batch step per iteration)
True generating line is θ = [4, 3]. GD matches the normal equation,
so it found the least-squares fit. Noise keeps that fit a bit off [4, 3].
Saved plots/assignment_5/q1_batch_gd_mse.png


**Q1 — MSE vs iteration**

![Batch GD MSE](../plots/assignment_5/q1_batch_gd_mse.png)


## Q2: Stochastic Gradient Descent vs Batch GD


In [3]:
section("Q2: Stochastic Gradient Descent")

# One random instance per step. Learning rate decays: η = t0 / (t + t1).
t0, t1 = 5, 50
n_epochs_sgd = 50
sgd_rng = np.random.RandomState(42)

theta = theta0.copy()
sgd_mse = []
sgd_updates_at = []
t = 0
for epoch in range(n_epochs_sgd):
    for _ in range(m):
        idx = sgd_rng.randint(m)
        xi = X_b[idx:idx + 1]
        yi = y[idx:idx + 1]
        gradients = 2 * xi.T @ (xi @ theta - yi)
        eta_t = t0 / (t + t1)
        theta = theta - eta_t * gradients
        t += 1
    sgd_mse.append(mse(X_b, y, theta))
    sgd_updates_at.append(t)

theta_sgd = theta
batch_updates = n_iterations
sgd_updates = n_epochs_sgd * m

print(f"SGD final θ:   θ0 = {theta_sgd[0, 0]:+.4f}   θ1 = {theta_sgd[1, 0]:+.4f}")
print(f"Final training MSE after {n_epochs_sgd} epochs: {sgd_mse[-1]:.4f}")
print()
print(f"{'Method':<22} {'Parameter updates':>20}")
print("-" * 44)
print(f"{'Batch GD':<22} {batch_updates:>20,}")
print(f"{'Stochastic GD':<22} {sgd_updates:>20,}")
print()
print("Batch GD: 1,000 iterations × 1 update = 1,000 updates.")
print("SGD: 50 epochs × 100 instances = 5,000 updates.")
print("SGD's path is noisier, but it reaches a similar θ with a decaying step size.")

plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, n_iterations + 1), batch_mse, color="steelblue", label="Batch GD")
plt.plot(
    sgd_updates_at, sgd_mse, color="darkorange", marker="o", ms=4,
    label="SGD (MSE after each epoch)",
)
plt.xlabel("Parameter updates")
plt.ylabel("Training MSE")
plt.title("MSE trajectory: Batch GD vs SGD")
plt.legend()
save_plot("q2_batch_vs_sgd_mse.png")



  Q2: Stochastic Gradient Descent
SGD final θ:   θ0 = +4.2151   θ1 = +2.7535
Final training MSE after 50 epochs: 0.8069

Method                    Parameter updates
--------------------------------------------
Batch GD                              1,000
Stochastic GD                         5,000

Batch GD: 1,000 iterations × 1 update = 1,000 updates.
SGD: 50 epochs × 100 instances = 5,000 updates.
SGD's path is noisier, but it reaches a similar θ with a decaying step size.
Saved plots/assignment_5/q2_batch_vs_sgd_mse.png


**Q2 — Batch GD vs SGD**

![Batch vs SGD](../plots/assignment_5/q2_batch_vs_sgd_mse.png)


## Q3: Mini-batch GD and comparison


In [4]:
section("Q3: Mini-batch GD")

# Q3 does not set a separate learning rate. Mini-batch uses the same
# decaying schedule as SGD: η = 5 / (t + 50). Batch size 20, 50 epochs.
batch_size = 20
n_epochs_mb = 50
mb_rng = np.random.RandomState(42)

theta = theta0.copy()
mb_mse = []
t = 0
for epoch in range(n_epochs_mb):
    order = mb_rng.permutation(m)
    for start in range(0, m, batch_size):
        batch_idx = order[start:start + batch_size]
        xi = X_b[batch_idx]
        yi = y[batch_idx]
        gradients = (2 / len(batch_idx)) * xi.T @ (xi @ theta - yi)
        eta_t = t0 / (t + t1)
        theta = theta - eta_t * gradients
        t += 1
    mb_mse.append(mse(X_b, y, theta))

theta_mb = theta
mb_updates = t

rows = [
    ("Batch GD", theta_batch, batch_mse, n_iterations, batch_updates),
    ("Stochastic GD", theta_sgd, sgd_mse, n_epochs_sgd, sgd_updates),
    ("Mini-batch GD", theta_mb, mb_mse, n_epochs_mb, mb_updates),
]

print(f"{'Variant':<16} {'θ0':>10} {'θ1':>10} {'epochs to MSE<1':>18} {'final MSE':>12} {'updates':>10}")
print("-" * 80)
for name, th, hist, n_ep, n_up in rows:
    hit = first_epoch_below(hist, 1.0)
    hit_txt = str(hit) if hit is not None else "not reached"
    print(f"{name:<16} {th[0, 0]:>10.4f} {th[1, 0]:>10.4f} {hit_txt:>18} {hist[-1]:>12.4f} {n_up:>10,}")

print()
print("Epoch = one pass over the 100 rows.")
print("Batch GD takes the most epochs but each step uses every row, so the path is smooth.")
print("SGD and mini-batch take fewer epochs because each epoch contains many smaller steps.")
print(f"Mini-batch updates = 50 epochs × (100/20) = {mb_updates}.")



  Q3: Mini-batch GD


Variant                  θ0         θ1    epochs to MSE<1    final MSE    updates
--------------------------------------------------------------------------------
Batch GD             4.2151     2.7701                 22       0.8066      1,000
Stochastic GD        4.2151     2.7535                  1       0.8069      5,000
Mini-batch GD        4.1349     2.8384                  6       0.8085        250

Epoch = one pass over the 100 rows.
Batch GD takes the most epochs but each step uses every row, so the path is smooth.
SGD and mini-batch take fewer epochs because each epoch contains many smaller steps.
Mini-batch updates = 50 epochs × (100/20) = 250.


## Q4: Polynomial regression and learning curves


In [5]:
section("Q4: Polynomial regression")

poly_rng = np.random.RandomState(42)
m_poly = 100
X_poly = 6 * poly_rng.rand(m_poly, 1) - 3
y_poly = 0.5 * X_poly**2 + X_poly + 2 + poly_rng.randn(*X_poly.shape)

X_line = np.linspace(-3, 3, 300).reshape(-1, 1)
degrees = [1, 2, 300]
colors = {1: "steelblue", 2: "seagreen", 300: "crimson"}
fitted = {}

print("True curve: y = 0.5 x² + x + 2 + noise")
print(f"{'Degree':>8} {'Train RMSE':>14} {'What it does':<28}")
print("-" * 54)
for degree in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lin", LinearRegression()),
    ])
    model.fit(X_poly, y_poly)
    fitted[degree] = model.predict(X_line)
    train_rmse = root_mean_squared_error(y_poly, model.predict(X_poly))
    note = {
        1: "underfits (straight line)",
        2: "matches the true degree",
        300: "overfits (explodes at edges)",
    }[degree]
    print(f"{degree:>8} {train_rmse:>14.4f} {note:<28}")

order = np.argsort(X_poly.ravel())
plt.figure(figsize=(8, 5))
plt.scatter(X_poly, y_poly, s=18, alpha=0.7, color="gray", label="Training data")
for degree in degrees:
    plt.plot(X_line, fitted[degree], color=colors[degree], lw=2, label=f"Degree {degree}")
plt.xlabel("X")
plt.ylabel("y")
plt.ylim(y_poly.min() - 2, y_poly.max() + 2)
plt.title("Polynomial fits: degree 1, 2, and 300")
plt.legend()
save_plot("q4_polynomial_fits.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, degree in zip(axes, [1, 300]):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lin", LinearRegression()),
    ])
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X_poly,
        y_poly.ravel(),
        train_sizes=np.linspace(0.15, 1.0, 8),
        cv=5,
        scoring="neg_root_mean_squared_error",
        shuffle=True,
        random_state=42,
    )
    train_rmse = -train_scores.mean(axis=1)
    val_rmse = -val_scores.mean(axis=1)
    ax.plot(train_sizes, train_rmse, "o-", color="steelblue", label="Train RMSE")
    ax.plot(train_sizes, val_rmse, "o-", color="darkorange", label="Validation RMSE")
    # Degree 300 validation error is orders of magnitude larger than training error.
    if degree == 300:
        ax.set_yscale("log")
    ax.set_xlabel("Training-set size")
    ax.set_ylabel("RMSE")
    ax.set_title(f"Learning curve — degree {degree}")
    ax.legend()
    print(f"\nDegree {degree}: final train RMSE {train_rmse[-1]:.3f}, validation RMSE {val_rmse[-1]:.3f}")

save_plot("q4_learning_curves.png")
print("\nDegree 1: train and validation RMSE stay close and high → underfitting.")
print("Degree 300: validation RMSE stays far above training RMSE → overfitting.")



  Q4: Polynomial regression
True curve: y = 0.5 x² + x + 2 + noise
  Degree     Train RMSE What it does                
------------------------------------------------------


       1         1.7391 underfits (straight line)   
       2         0.8816 matches the true degree     
     300         2.1980 overfits (explodes at edges)
Saved plots/assignment_5/q4_polynomial_fits.png

Degree 1: final train RMSE 1.738, validation RMSE 1.745



Degree 300: final train RMSE 2.164, validation RMSE 11525.192


Saved plots/assignment_5/q4_learning_curves.png

Degree 1: train and validation RMSE stay close and high → underfitting.
Degree 300: validation RMSE stays far above training RMSE → overfitting.


**Q4 — fitted curves and learning curves**

![Polynomial fits](../plots/assignment_5/q4_polynomial_fits.png)

![Learning curves](../plots/assignment_5/q4_learning_curves.png)


## Q5: Ridge, Lasso, ElasticNet on red wine quality


In [6]:
section("Q5: Ridge, Lasso, ElasticNet")

wine = pd.read_csv(wine_path, sep=";")
feature_names = [c for c in wine.columns if c != "quality"]
X_wine = wine[feature_names]
y_wine = wine["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)
print(f"Rows: {len(wine):,}   features: {len(feature_names)}")
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")
print("Target: quality (score 0–10). Features are scaled inside each pipeline.")

alphas = [0.001, 0.01, 0.1, 1, 10]
models = {
    "Ridge": Ridge,
    "Lasso": Lasso,
    "ElasticNet": ElasticNet,
}

records = []
lasso_coefs = []
for name, cls in models.items():
    for alpha in alphas:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", cls(alpha=alpha, max_iter=100000)),
        ])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rmse = root_mean_squared_error(y_test, pred)
        records.append({"model": name, "alpha": alpha, "test_rmse": rmse})
        if name == "Lasso":
            lasso_coefs.append(pipe.named_steps["model"].coef_.copy())

rmse_table = pd.DataFrame(records).pivot(index="model", columns="alpha", values="test_rmse")
rmse_table = rmse_table[alphas]
print("\nTest RMSE")
header = f"{'model':<14}" + "".join(f"{a:>10}" for a in alphas)
print(header)
print("-" * len(header))
for model_name, row in rmse_table.iterrows():
    print(f"{model_name:<14}" + "".join(f"{v:>10.4f}" for v in row))

best = min(records, key=lambda r: r["test_rmse"])
print(f"\nLowest test RMSE: {best['model']} alpha={best['alpha']}  RMSE={best['test_rmse']:.4f}")
print("ElasticNet uses l1_ratio=0.5 (half L1, half L2).")

coef_df = pd.DataFrame(lasso_coefs, columns=feature_names, index=alphas)
coef_df.index.name = "alpha"

# "Reaches zero first" = zero at the smallest alpha in this grid.
zero_tol = 1e-8
first_zero = {}
for feature in feature_names:
    for alpha in alphas:
        if abs(coef_df.loc[alpha, feature]) <= zero_tol:
            first_zero[feature] = alpha
            break

print("\nLasso: smallest alpha at which each coefficient is 0")
print(f"{'Feature':<24} {'first α with coef = 0':>24}")
print("-" * 50)
for feature in feature_names:
    alpha_hit = first_zero.get(feature)
    shown = f"{alpha_hit:g}" if alpha_hit is not None else "never on this grid"
    print(f"{feature:<24} {shown:>24}")

if first_zero:
    earliest = min(first_zero.values())
    winners = [f for f, a in first_zero.items() if a == earliest]
    print(f"\nOn the required grid, first zero is at alpha = {earliest:g}: {', '.join(winners)}")
else:
    print("\nNo Lasso coefficient is zero on this alpha grid.")

# The grid is coarse, so break the tie with a finer alpha path.
fine_alphas = np.linspace(0.001, 0.01, 37)
fine_hit = {}
for alpha in fine_alphas:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=float(alpha), max_iter=100000)),
    ])
    pipe.fit(X_train, y_train)
    for feature, coef in zip(feature_names, pipe.named_steps["model"].coef_):
        if feature not in fine_hit and abs(coef) <= zero_tol:
            fine_hit[feature] = float(alpha)
first_feature = min(fine_hit, key=fine_hit.get)
print(
    f"Finer path: {first_feature} reaches zero first, at alpha ≈ {fine_hit[first_feature]:.4f}."
)
order = sorted(fine_hit, key=fine_hit.get)[:3]
print("Next:", ", ".join(f"{f} (α≈{fine_hit[f]:.4f})" for f in order[1:]))

plt.figure(figsize=(9, 6))
for feature in feature_names:
    plt.plot(alphas, coef_df[feature], marker="o", label=feature)
plt.xscale("log")
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("alpha")
plt.ylabel("Lasso coefficient (after scaling)")
plt.title("Lasso coefficients vs alpha")
plt.legend(fontsize=8, loc="best")
save_plot("q5_lasso_coefficients.png")



  Q5: Ridge, Lasso, ElasticNet
Rows: 1,599   features: 11
Train: 1,279   Test: 320
Target: quality (score 0–10). Features are scaled inside each pipeline.

Test RMSE
model              0.001      0.01       0.1         1        10
----------------------------------------------------------------
ElasticNet        0.6247    0.6263    0.6441    0.8107    0.8107
Lasso             0.6249    0.6269    0.6627    0.8107    0.8107
Ridge             0.6245    0.6245    0.6245    0.6245    0.6246

Lowest test RMSE: Ridge alpha=0.001  RMSE=0.6245
ElasticNet uses l1_ratio=0.5 (half L1, half L2).

Lasso: smallest alpha at which each coefficient is 0
Feature                     first α with coef = 0
--------------------------------------------------
fixed acidity                                 0.1
volatile acidity                                1
citric acid                                  0.01
residual sugar                               0.01
chlorides                                     0.1
free

Saved plots/assignment_5/q5_lasso_coefficients.png


**Q5 — Lasso coefficients vs alpha**

![Lasso coefficients](../plots/assignment_5/q5_lasso_coefficients.png)


## Q6: Logistic regression — good wine vs not


In [7]:
section("Q6: Logistic regression (quality ≥ 7)")

y_good = (wine["quality"] >= 7).astype(int)
print("Positive class: good (quality ≥ 7)")
print(y_good.value_counts().rename({0: "not good", 1: "good"}).to_string())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_wine, y_good, test_size=0.2, random_state=42, stratify=y_good
)
logit = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
logit.fit(X_tr, y_tr)
y_hat = logit.predict(X_te)

acc = accuracy_score(y_te, y_hat)
prec = precision_score(y_te, y_hat, zero_division=0)
rec = recall_score(y_te, y_hat, zero_division=0)
f1 = f1_score(y_te, y_hat, zero_division=0)

print(f"\nTrain: {len(X_tr):,}   Test: {len(X_te):,}   (stratified 80/20)")
print(f"{'Accuracy':<12} {acc:.4f}   ({100 * acc:.2f}%)")
print(f"{'Precision':<12} {prec:.4f}   (of predicted good, fraction truly good)")
print(f"{'Recall':<12} {rec:.4f}   (of truly good, fraction caught)")
print(f"{'F1':<12} {f1:.4f}")
print("\nGood wines are the minority, so accuracy stays high even if recall is modest.")



  Q6: Logistic regression (quality ≥ 7)
Positive class: good (quality ≥ 7)
quality
not good    1382
good         217

Train: 1,279   Test: 320   (stratified 80/20)
Accuracy     0.8938   (89.38%)
Precision    0.6957   (of predicted good, fraction truly good)
Recall       0.3721   (of truly good, fraction caught)
F1           0.4848

Good wines are the minority, so accuracy stays high even if recall is modest.


## Q7: Iris multinomial logistic regression


In [8]:
section("Q7: Iris decision boundary")

# sklearn 1.8+ dropped multi_class. LogisticRegression with lbfgs fits the
# multinomial (softmax) loss for 3 classes, which is what multi_class="multinomial" did.
iris = load_iris()
X_iris = iris.data[:, 2:4]
y_iris = iris.target
petal_names = ["petal length (cm)", "petal width (cm)"]

clf = LogisticRegression(max_iter=1000)
clf.fit(X_iris, y_iris)
train_acc = accuracy_score(y_iris, clf.predict(X_iris))
print("Features: petal length, petal width")
print(f"Classes: {list(iris.target_names)}")
print(f"Training accuracy: {train_acc:.4f} ({100 * train_acc:.2f}%)")
print("Decision boundary is the set of points where two class probabilities are equal.")

x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400),
)
grid = np.c_[xx.ravel(), yy.ravel()]
zz = clf.predict(grid).reshape(xx.shape)

bg = ListedColormap(["#c7e9c0", "#fdd0a2", "#c6dbef"])
pt = ListedColormap(["#238b45", "#d94801", "#2171b5"])

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5, 2.5], cmap=bg, alpha=0.9)
for class_id, name in enumerate(iris.target_names):
    mask = y_iris == class_id
    plt.scatter(
        X_iris[mask, 0],
        X_iris[mask, 1],
        s=28,
        color=pt(class_id),
        edgecolor="white",
        linewidth=0.4,
        label=name,
    )
plt.xlabel(petal_names[0])
plt.ylabel(petal_names[1])
plt.title("Iris: multinomial logistic regression")
plt.legend(title="Species")
save_plot("q7_iris_decision_boundary.png")



  Q7: Iris decision boundary
Features: petal length, petal width
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Training accuracy: 0.9667 (96.67%)
Decision boundary is the set of points where two class probabilities are equal.
Saved plots/assignment_5/q7_iris_decision_boundary.png


**Q7 — decision boundary**

![Iris decision boundary](../plots/assignment_5/q7_iris_decision_boundary.png)
